# 02 Data quality and cleaning

**Purpose:** check every table against the rules it should follow, explain anything unusual, and save clean, typed tables for the rest of the project.

**Inputs:** the raw data files in `data/raw/`.

**Outputs** (in `data/processed/`):
- `orders.parquet`: train and test orders in one table, with a `split` column
- `legs.parquet`: train and test route legs in one table
- `outlets.parquet`, `vehicles.parquet`, `calendar.parquet`, `road_conditions.parquet`, `traffic_speed.parquet`, `district_travel.parquet`, `service_allowance.parquet`

**Approach:** every rule is written as a check that reports how many rows break it. One summary table shows everything at once, instead of the notebook stopping at the first problem. Nothing is dropped silently: each fix or exclusion is stated with its reason.

In [1]:
import numpy as np
import pandas as pd

from xora.io import load_raw, save_processed
from xora.timeutils import hhmm_to_minutes
from xora import checks as ck

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

In [2]:
orders = load_raw("deliveries_train.csv")
legs = load_raw("route_legs_train.csv")
orders_test = load_raw("task1_test_inputs.csv")
legs_test = load_raw("route_legs_test.csv")
outlets = load_raw("outlets.csv")
vehicles = load_raw("vehicles.csv")
calendar = load_raw("calendar.csv")
road = load_raw("road_conditions.csv")
traffic = load_raw("traffic_speed.csv")
district_travel = load_raw("district_travel.csv")
allowance = load_raw("service_allowance.csv")

## 1. Structural checks

Keys must be unique, categories must take only their documented values, and quantities must be positive.

In [3]:
BRANDS = {"Fresh", "Style", "Tech"}
DEPOTS = {"Peliyagoda", "Kandy"}

rows = []
for name, df in [("orders train", orders), ("orders test", orders_test)]:
    rows += [
        ck.unique_key(df, ["delivery_id"], name),
        ck.allowed_values(df, "brand", BRANDS, name),
        ck.allowed_values(df, "depot", DEPOTS, name),
        ck.allowed_values(df, "temp_requirement", {"chilled", "ambient"}, name),
        ck.allowed_values(df, "dispatch_status", {"attempted", "deferred", "not_run"}, name),
        ck.check(f"{name}: units, weight and volume are positive",
                 (df[["order_units", "order_weight_kg", "order_volume_m3"]] <= 0).any(axis=1).sum(), len(df)),
    ]
for name, df in [("legs train", legs), ("legs test", legs_test)]:
    rows += [
        ck.unique_key(df, ["leg_id"], name),
        ck.unique_key(df, ["route_id", "seq"], name),
        ck.in_range(df, "dow", 0, 5, name),
        ck.in_range(df, "monsoon", 0, 1, name),
        ck.check(f"{name}: distance is positive", (df.distance_km <= 0).sum(), len(df)),
    ]
rows += [
    ck.unique_key(outlets, ["outlet_id"], "outlets"),
    ck.unique_key(vehicles, ["vehicle_id"], "vehicles"),
    ck.unique_key(calendar, ["date"], "calendar"),
    ck.unique_key(road, ["district", "date"], "road conditions"),
    ck.check("train and test share no delivery_id", len(set(orders.delivery_id) & set(orders_test.delivery_id)), len(orders_test)),
    ck.check("test and train share no route_id", legs_test.route_id.isin(legs.route_id).sum(), len(legs_test)),
]
ck.report(rows)

,check,passed,failing_rows,total_rows,note
0,orders train: delivery_id is unique,True,0,92307,
1,"orders train: brand in ['Fresh', 'Style', 'Tech']",True,0,92307,
2,"orders train: depot in ['Kandy', 'Peliyagoda']",True,0,92307,
3,"orders train: temp_requirement in ['ambient', 'chilled']",True,0,92307,
4,"orders train: dispatch_status in ['attempted', 'deferred', 'not_run']",True,0,92307,
5,"orders train: units, weight and volume are positive",True,0,92307,
6,orders test: delivery_id is unique,True,0,5014,
7,"orders test: brand in ['Fresh', 'Style', 'Tech']",True,0,5014,
8,"orders test: depot in ['Kandy', 'Peliyagoda']",True,0,5014,
9,"orders test: temp_requirement in ['ambient', 'chilled']",True,0,5014,


Every structural check passes. Day of week never reaches 6 because Waypoint does not deliver on Sundays.

## 2. Blanks and cross-table consistency

Orders repeat attributes that also live in the reference tables (an outlet's brand and window, a vehicle's type). If the copies disagreed we would have to pick one, so we check they match.

In [4]:
orders.isna().sum().loc[lambda s: s > 0].rename("blank values").to_frame().assign(
    not_run_orders=(orders.dispatch_status == "not_run").sum()
)

,blank values,not_run_orders
dispatch_date,413,413
route_id,413,413
seq_in_route,413,413
vehicle_id,413,413
vehicle_type,413,413
vehicle_temp,413,413
planned_arrival_time,413,413


The only blanks are the route fields of the 413 `not_run` orders. These orders were never dispatched, so there is nothing to fill. They stay in the data because Task 2A must count every order, but they cannot be used for Task 1 labels.

In [5]:
both = pd.concat([orders, orders_test], ignore_index=True)

with_outlet = both.merge(outlets, on="outlet_id", suffixes=("", "_ref"))
with_vehicle = both.dropna(subset=["vehicle_id"]).merge(vehicles, on="vehicle_id", suffixes=("", "_ref"))

rows = [
    ck.check(f"order {col} matches outlets.csv", (with_outlet[col] != with_outlet[col + "_ref"]).sum(), len(with_outlet))
    for col in ["brand", "district", "depot", "window_open_time", "window_close_time"]
]
rows += [
    ck.check("order vehicle_type matches vehicles.csv", (with_vehicle.vehicle_type != with_vehicle.type).sum(), len(with_vehicle)),
    ck.check("order vehicle_temp matches vehicles.csv", (with_vehicle.vehicle_temp != with_vehicle.temp).sum(), len(with_vehicle)),
    ck.check("vehicle serves its own depot", (with_vehicle.depot != with_vehicle.depot_ref).sum(), len(with_vehicle)),
    ck.check("chilled orders ride on reefers", ((both.temp_requirement == "chilled") & (both.vehicle_temp == "ambient")).sum(), len(both)),
    ck.check("van-only outlets get vans",
             ((with_outlet.parking_constraint == "van_only") & (with_outlet.vehicle_type == "truck")).sum(), len(with_outlet)),
]
mall = outlets[outlets.mall_window.notna()]
rows.append(ck.check("mall window equals the outlet window", (mall.mall_window != mall.window_open_time + "-" + mall.window_close_time).sum(), len(mall)))
rows.append(ck.check("orders fall on operating days",
                     both.merge(calendar, left_on="order_date", right_on="date").is_operating.eq(0).sum(), len(both)))
ck.report(rows)

,check,passed,failing_rows,total_rows,note
0,order brand matches outlets.csv,True,0,97321,
1,order district matches outlets.csv,True,0,97321,
2,order depot matches outlets.csv,True,0,97321,
3,order window_open_time matches outlets.csv,True,0,97321,
4,order window_close_time matches outlets.csv,True,0,97321,
5,order vehicle_type matches vehicles.csv,True,0,96908,
6,order vehicle_temp matches vehicles.csv,True,0,96908,
7,vehicle serves its own depot,True,0,96908,
8,chilled orders ride on reefers,True,0,97321,
9,van-only outlets get vans,True,0,97321,


The copies agree everywhere, and the historical plans respected the operating rules (reefers for chilled, vans for van-only outlets, own depot). For mall outlets, the outlet window already equals the mall window, so the label code needs only one window.

## 3. Time consistency on the route legs

Each leg should follow the same logic: leave the previous point, drive, arrive, unload, leave. If these times hang together, we can trust labels built from them.

In [6]:
TIME_COLS = ["planned_depart_time", "planned_arrival_time", "actual_depart_time", "arrival_time", "leave_outlet_time"]
for col in TIME_COLS:
    legs[col + "_min"] = hhmm_to_minutes(legs[col])

legs = legs.sort_values(["route_id", "seq"]).reset_index(drop=True)
by_route = legs.groupby("route_id")
next_depart = by_route.actual_depart_time_min.shift(-1)
prev_outlet = by_route.to_outlet.shift()
gap = legs.arrival_time_min - (legs.actual_depart_time_min + legs.actual_travel_duration_min)

rows = [
    ck.check("routes start at seq 0 from the depot", (legs.loc[legs.seq == 0, "from_point"] != "DEPOT").sum(), (legs.seq == 0).sum()),
    ck.check("each leg starts where the last one ended", (legs.from_point[legs.seq > 0] != prev_outlet[legs.seq > 0]).sum(), (legs.seq > 0).sum()),
    ck.check("one vehicle, date, brand and district per route",
             (by_route[["vehicle_id", "date", "brand", "district"]].nunique() > 1).any(axis=1).sum(), legs.route_id.nunique()),
    ck.check("planned arrival = planned departure + planned travel",
             (legs.planned_arrival_time_min != legs.planned_depart_time_min + legs.planned_travel_duration_min).sum(), len(legs)),
    ck.check("actual arrival = actual departure + travel, within 1 minute", (gap.abs() > 1).sum(), len(legs),
             "differences of exactly 1 minute come from rounding to HH:MM"),
    ck.check("vehicle leaves after it arrives", (legs.leave_outlet_time_min < legs.arrival_time_min).sum(), len(legs)),
    ck.check("next leg departs exactly when the vehicle leaves", (next_depart.notna() & (next_depart != legs.leave_outlet_time_min)).sum(), next_depart.notna().sum()),
    ck.check("all times fall within the same day", (legs[[c + "_min" for c in TIME_COLS]] >= 24 * 60).any(axis=1).sum(), len(legs)),
]
ck.report(rows)

,check,passed,failing_rows,total_rows,note
0,routes start at seq 0 from the depot,True,0,25198,
1,each leg starts where the last one ended,True,0,66696,
2,"one vehicle, date, brand and district per route",True,0,25198,
3,planned arrival = planned departure + planned travel,True,0,91894,
4,"actual arrival = actual departure + travel, within 1 minute",True,0,91894,differences of exactly 1 minute come from rounding to HH:MM
5,vehicle leaves after it arrives,True,0,91894,
6,next leg departs exactly when the vehicle leaves,True,0,66696,
7,all times fall within the same day,True,0,91894,


In [7]:
gap.value_counts().sort_index().rename("legs").rename_axis("arrival minus (departure + travel), minutes").to_frame().T

"arrival minus (departure + travel), minutes",-1.0,0.0,1.0
legs,11518,68820,11556


Every rule holds. The only wrinkle is rounding: times are stored to the minute, so in about a quarter of legs the arrival is one minute off departure plus travel. This means every label we build carries roughly one minute of noise, which is worth remembering when reading model errors.